In [ ]:
import random
from datasets import load_dataset, Dataset
import os

# ==============================================================================
# 💡 데이터셋 소개: Magpie-Qwen2-Pro-200K-English-ko
# ✨ 대략적 의미: "창조적 대화 데이터셋 큐레이션 실습"
# 📘 설명: 이 데이터셋은 '질문-답변' 형태의 대규모 대화(Instruction/Response) 데이터를 담고 있습니다.
#        특히, 단순히 질문만 하는 것이 아니라, 사용자의 의도(intent), 난이도(difficulty),
#        태스크 카테고리(task_category) 등 다양한 메타데이터와 함께 제공되어,
#        어떤 종류의 AI를 만들지 '데이터를 선별'하는(Curating) 방법을 배우기에 최고입니다!
# ==============================================================================

# 상수 설정
DATASET_NAME = "youjunhyeok/Magpie-Qwen2-Pro-200K-English-ko"
SPLIT_NAME = 'train'
SAMPLE_COUNT = 10  # 분석할 샘플 개수 (너무 많으면 실행 시간이 길어요!)

# ------------------------------------------------------------------------------
# 🚀 1단계: 데이터 로드 및 스트리밍 확인 (가장 중요해요!)
# ------------------------------------------------------------------------------

dataset = None
sample_data_list = []

print("⭐ [튜터의 속삭임] 데이터셋을 로드하기 전에, 크고 무거운 데이터셋을 효율적으로 다루는 방법을 배워볼 거예요. '스트리밍(streaming)' 기능을 사용해서요!")

try:
    # 1. 스트리밍 모드로 로드 시도 (가장 빠르고 메모리 효율적!)
    print(f"\n✅ 시도: 스트리밍 모드 ({DATASET_NAME}, {SPLIT_NAME}, streaming=True) 로 데이터셋 연결 중...")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("🎉 성공! 스트리밍 데이터셋을 성공적으로 가져왔어요. 메모리 걱정 NO!")

except Exception as e:
    # 2. 스트리밍 실패 시, 일반 데이터셋으로 로드 (최후의 보루!)
    print(f"\n⚠️ 경고: 스트리밍 로드 중 오류 발생 ({e}). 일반 Dataset 모드로 전환합니다.")
    try:
        # 테스트용으로 작은 샘플만 로드하여 중단 없이 진행
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
        print(f"✅ 성공: 일반 Dataset 모드로 {DATASET_NAME}을 로드했습니다.")
    except Exception as e_fallback:
        print(f"🚨 치명적인 오류: 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()

# ------------------------------------------------------------------------------
# 💻 2단계: 데이터 샘플링 패턴 적용 (핵심 기술!)
# ------------------------------------------------------------------------------

print("\n======================================================================")
print("🧠 2단계: 데이터 샘플링 (Sampling) 과정 - 100만개 중 10개만 골라보기!")
print("======================================================================")

if hasattr(dataset, "take"):
    # .take()가 존재하면 -> 스트리밍 데이터셋(IterableDataset) 패턴
    print("✨ Detected: Streaming Mode. .take() 함수를 사용합니다.")
    # 다음 패턴 사용:
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
    
    # 반복 가능한 이터레이터에서 리스트로 변환 (가장 안전한 방법!)
    sample_data_list = []
    for _ in range(SAMPLE_COUNT):
        try:
            sample_data_list.append(next(sampled_dataset_iterator))
        except StopIteration:
            break # 데이터가 부족하면 멈춥니다.
else:
    # 일반 데이터셋 (Dataset) 패턴
    print("✨ Detected: Full Dataset Mode. 일반 샘플링 패턴을 사용합니다.")
    # 일반 패턴 사용:
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
    
print(f"💡 준비 완료! 총 {len(sample_data_list)}개의 샘플을 분석할 준비가 되었습니다.")


# ------------------------------------------------------------------------------
# 🤖 3단계: 창의적 실습 - 고품질 프롬프트 큐레이터 되기!
# ------------------------------------------------------------------------------
# 목표: 주어진 샘플들을 분석하여, 실제 AI 모델을 학습시키기 위한 '프롬프트 구조'를 디자인하는 시뮬레이션.

print("\n" + "="*90)
print("✨ 🌟 3단계 실습: 고품질 데이터 선별 및 프롬프트 구조화 🌟 ✨")
print("======================================================================")

for i, sample in enumerate(sample_data_list):
    print(f"\n\n===== [🔬 샘플 번호 {i+1}/{len(sample_data_list)}] 분석 시작 =====")

    # 1. 메타데이터 분석: 어떤 종류의 데이터를 다루는지 파악합니다. (정량적 분석!)
    task = sample.get('task_category', 'N/A')
    difficulty = sample.get('difficulty', 'N/A')
    intent = sample.get('intent', 'N/A')
    
    print(f"\n[📂 1. 메타데이터 분석 결과]")
    print(f"  - 💡 태스크 카테고리: {task} (이 샘플은 '{task}'와 관련된 작업을 합니다.)")
    print(f"  - 🧐 예상 난이도: {difficulty}")
    print(f"  - 🎯 사용자 의도 (Intent): {intent}")

    # 2. 대화 구조 확인: conversations 필드를 통해 상호작용 패턴을 이해합니다.
    conversations = sample.get('conversations', [])
    if conversations:
        # 대화 턴이 있을 경우, 첫 번째 턴만 간략히 구조를 보여줍니다.
        print(f"[🗣️ 2. 대화 구조 분석 (Conversation Flow)]")
        print(f"  - 대화 턴의 수: {len(conversations)}회")
        # 첫 번째 턴의 화자와 내용을 뽑아 구조를 보여줍니다.
        first_turn = conversations[0]
        print(f"  -  첫 번째 발화 구조 (From/Value): '{first_turn['from']}'가 '{first_turn['value'][:30]}...' 라고 말했네요.")
    
    # 3. 프롬프트 생성 시뮬레이션: 최종적으로 AI에게 던질 '프롬프트'를 구성합니다.
    print("\n[💻 3. 모델 학습용 프롬프트 템플릿 생성 시뮬레이션]")
    
    # 사용자가 추구해야 할 핵심 요소들을 구조화하여 출력합니다.
    prompt_template = f"""
    [SYSTEM PROMPT]
    당신은 {task}와 관련된 질문에 전문적으로 답변하는 AI입니다.
    지시사항: '{sample.get('instruction', 'N/A')}' 이라는 지시를 따르세요.

    [USER INPUT]
    질문: {sample.get('instruction', 'N/A')}

    [MODEL RESPONSE]
    답변: {sample.get('response', 'N/A')}
    """
    print("-" * 50)
    print(">>> 이 템플릿 구조를 사용해 데이터를 학습시키면 가장 효과적입니다.")
    print(prompt_template.strip())
    print("-" * 50)

print("\n\n======================================================================")
print("🎉 축하합니다! ✨ 데이터를 분석하고, 적절한 프롬프트 구조를 디자인하는 과정을 성공적으로 마쳤어요.")
print("🤖 이제 당신은 단순한 데이터 사용자가 아니라, 데이터를 선별하고 모델을 '훈련시키는 아키텍트'입니다!")
print("======================================================================")